# HiNet: Deep Image Hiding by Invertible Network

**ICCV 2021** — Restructured with Nexus-Steg pipeline

This notebook is designed for **Google Colab**. It will:
1. Clone the repository
2. Download the DIV2K dataset to `/content/`
3. Run sanity check (Karpathy Recipe)
4. Overfit one batch (capacity test)
5. Full training
6. Evaluation with attack robustness tests

## 0. GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

## 1. Clone Repository

In [ ]:
import os
os.chdir("/content")

if not os.path.exists("/content/HiNet"):
    !git clone https://github.com/raghav-potdar/HiNet.git
else:
    print("Repository already cloned — pulling latest")
    !cd /content/HiNet && git pull

os.chdir("/content/HiNet")
print(f"Working directory: {os.getcwd()}")

## 2. Download DIV2K Dataset

Downloads **DIV2K_train_HR** (800 images) and **DIV2K_valid_HR** (100 images) to `/content/datasets/`.
Skips if already downloaded.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

DATASET_DIR = Path("/content/HiNet/datasets")
DATASET_DIR.mkdir(exist_ok=True)

URLS = {
    "DIV2K_train_HR": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip",
    "DIV2K_valid_HR": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip",
}


def download_and_extract(name, url, dest):
    target = dest / name
    if target.exists() and any(target.glob("*.png")):
        count = len(list(target.glob("*.png")))
        print(f"[{name}] Already exists with {count} images — skipping")
        return

    zip_path = dest / f"{name}.zip"
    print(f"[{name}] Downloading from {url} ...")

    def _progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = min(100, downloaded * 100 / total_size)
            mb = downloaded / 1e6
            total_mb = total_size / 1e6
            print(f"\r  {pct:.0f}% ({mb:.0f}/{total_mb:.0f} MB)", end="", flush=True)

    urllib.request.urlretrieve(url, zip_path, reporthook=_progress)
    print(f"\n[{name}] Extracting ...")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest)
    zip_path.unlink()

    count = len(list(target.glob("*.png")))
    print(f"[{name}] Done — {count} images")


for name, url in URLS.items():
    download_and_extract(name, url, DATASET_DIR)

print(f"\nDataset ready at {DATASET_DIR}/")

In [ ]:
# Quick peek at a few images
import matplotlib.pyplot as plt
from PIL import Image

train_imgs = sorted((DATASET_DIR / "DIV2K_train_HR").glob("*.png"))[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, train_imgs):
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(p.name, fontsize=9)
    ax.axis("off")
fig.suptitle("DIV2K Train Samples", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Sanity Check (Karpathy Recipe — Step 1)

Verifies:
- Data pipeline loads images correctly
- Initial loss values are in expected range
- Saves `results/sanity_inputs.png` for visual inspection

In [ ]:
os.chdir("/content/HiNet")
!python main.py --sanity --train_dir /content/datasets/DIV2K_train_HR --val_dir /content/datasets/DIV2K_valid_HR

In [ ]:
from IPython.display import display, Image as IPImage

sanity_path = "/content/HiNet/results/sanity_inputs.png"
if os.path.exists(sanity_path):
    print("Top row: Cover images | Bottom row: Secret images")
    display(IPImage(filename=sanity_path))
else:
    print("Sanity check image not found — run the cell above first")

## 4. Overfit One Batch (Karpathy Recipe — Step 2)

Trains on a single batch for 500 steps to verify the model has enough capacity.

| Result | r_loss | Meaning |
|--------|--------|---------|
| **PASS** | < 0.01 | Model can memorize — proceed to training |
| **WARN** | 0.01 - 0.10 | Likely fine, monitor during training |
| **FAIL** | > 0.10 | Architecture or LR problem — do not proceed |

In [ ]:
os.chdir("/content/HiNet")
!python main.py --overfit_one_batch --train_dir /content/datasets/DIV2K_train_HR --val_dir /content/datasets/DIV2K_valid_HR

## 5. Full Training

Two-phase schedule:
- **Phase 1** (epochs 0-49): Pure hiding/recovery, no noise
- **Phase 2** (epochs 50+): Noise augmentation active for robustness

Adjust `--epochs` and `--batch_size` for your Colab GPU:
- **T4 (free tier):** `--batch_size 8 --epochs 100`
- **A100 (Colab Pro):** `--batch_size 16 --epochs 100`

In [ ]:
os.chdir("/content/HiNet")
!python main.py \
    --epochs 100 \
    --batch_size 16 \
    --checkpoint_every 10 \
    --patience 15 \
    --train_dir /content/datasets/DIV2K_train_HR \
    --val_dir /content/datasets/DIV2K_valid_HR

In [ ]:
# Plot training curves from CSV log
import csv
import matplotlib.pyplot as plt

csv_path = "/content/HiNet/results/training_log.csv"
if not os.path.exists(csv_path):
    print("No training log found — run training first")
else:
    epochs, train_loss, psnr_stego, psnr_secret, ssim_secret = [], [], [], [], []
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs.append(int(row["epoch"]))
            train_loss.append(float(row["train_loss"]))
            psnr_stego.append(float(row["val_psnr_stego"]))
            psnr_secret.append(float(row["val_psnr_secret"]))
            ssim_secret.append(float(row["val_ssim_secret"]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, train_loss)
    axes[0].set_title("Train Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].plot(epochs, psnr_stego, label="PSNR(stego)")
    axes[1].plot(epochs, psnr_secret, label="PSNR(secret)")
    axes[1].set_title("Validation PSNR")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("dB")
    axes[1].legend()

    axes[2].plot(epochs, ssim_secret)
    axes[2].set_title("SSIM(secret)")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("SSIM")

    plt.tight_layout()
    plt.show()

In [ ]:
# Show visual results from the latest epoch
import glob
from IPython.display import display, Image as IPImage

epoch_imgs = sorted(glob.glob("/content/HiNet/results/epoch_*.png"))
if epoch_imgs:
    last = epoch_imgs[-1]
    print(f"Showing: {last}  (Cover | Secret | Stego | Revealed)")
    display(IPImage(filename=last))
else:
    print("No epoch images found — run training first")

## 6. Evaluation

Runs 8 attack tests on the best checkpoint:
- Clean (no attack)
- JPEG compression (Q=90, Q=50)
- Gaussian blur
- Gaussian noise
- Resize (50%, 75%)
- Social media simulation (resize + JPEG)

In [ ]:
os.chdir("/content/HiNet")
!python evaluate.py \
    --checkpoint checkpoints/hinet_best.pth \
    --val_dir /content/datasets/DIV2K_valid_HR

In [ ]:
# Show evaluation report
report_path = "/content/HiNet/results/evaluation/report.txt"
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())
else:
    print("No report found — run evaluation first")

In [ ]:
# Show attack result images
import glob
from IPython.display import display, Image as IPImage

attack_imgs = sorted(glob.glob("/content/HiNet/results/evaluation/*.png"))
if attack_imgs:
    for img_path in attack_imgs:
        name = os.path.basename(img_path).replace(".png", "")
        print(f"\n--- {name} ---  (Cover | Secret | Stego | Attacked | Revealed)")
        display(IPImage(filename=img_path, width=800))
else:
    print("No evaluation images found — run evaluation first")

## 7. Save Checkpoints to Google Drive (Optional)

Colab VMs are ephemeral — save your checkpoints to Drive before the session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil

DRIVE_SAVE = "/content/drive/MyDrive/HiNet_checkpoints"
os.makedirs(DRIVE_SAVE, exist_ok=True)

for ckpt in glob.glob("/content/HiNet/checkpoints/*.pth"):
    dest = os.path.join(DRIVE_SAVE, os.path.basename(ckpt))
    shutil.copy2(ckpt, dest)
    print(f"Saved: {dest}")

# Also save training log and results
for f in ["/content/HiNet/results/training_log.csv",
          "/content/HiNet/results/evaluation/report.txt"]:
    if os.path.exists(f):
        shutil.copy2(f, DRIVE_SAVE)
        print(f"Saved: {os.path.join(DRIVE_SAVE, os.path.basename(f))}")

print(f"\nAll saved to {DRIVE_SAVE}")